In [10]:
!pip install torch transformers accelerate peft datasets trl bitsandbytes sacremoses

In [11]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import re

assert torch.cuda.is_available(), "CUDA not available. Please enable GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [12]:
!pip install --upgrade torchao>=0.16.0

In [13]:
# Model Loading & LoRA Configuration
model_name = "microsoft/biogpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# BioGPT tokenizer has no pad_token; set it to eos_token for batch processing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in FP16 to save memory
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Freeze base model and add LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                     # rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],   # BioGPT (GPT-2 style) attention modules
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 786,432 || all params: 390,950,912 || trainable%: 0.2012


In [14]:
# Load and format dataset
dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
dataset = dataset.select(range(2000))   # use first 2000 samples
print(f"Total training samples: {len(dataset)}")

system_prompt = "You are a family doctor. Give concise and practical advice."

def format_chat(example):
    convs = example["conversations"]
    # Verify structure (safety check)
    if len(convs) != 2 or convs[0]["from"] != "human" or convs[1]["from"] != "gpt":
        return None
    user_msg = convs[0]["value"].strip()
    assistant_msg = convs[1]["value"].strip()
    # Remove common greetings from assistant's response
    assistant_clean = re.sub(r"^(Hello|Hi|Thank you for|Welcome to).*?\.\s*", "", assistant_msg, flags=re.IGNORECASE)
    assistant_clean = assistant_clean[:300]   # keep short
    text = f"{system_prompt}\nUser: {user_msg}\nAssistant: {assistant_clean}\n"
    return {"text": text}

# Apply formatting and remove any None entries
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: x["text"] is not None)
print(f"After cleaning: {len(dataset)} samples")

Total training samples: 2000
After cleaning: 2000 samples


In [15]:
# Tokenization
def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenization done. Example shape:", tokenized_dataset[0]["input_ids"].shape)

Tokenization done. Example shape: torch.Size([512])


In [ ]:
# 5. Training
training_args = TrainingArguments(
    output_dir="./biogpt-familydoctor",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

# Save LoRA weights and tokenizer
model.save_pretrained("./biogpt-familydoctor-final")
tokenizer.save_pretrained("./biogpt-familydoctor-final")
print("Training completed and model saved.")

Step,Training Loss
50,4.315160


In [ ]:
# Test the fine‑tuned model
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Reload model and LoRA (if not already in memory)
base_model_name = "microsoft/biogpt"
lora_path = "./biogpt-medical-qa-lora-final"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
base_model.config.tie_word_embeddings = False
model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()
print("Model loaded successfully.")

def generate_answer(question, max_new_tokens=150, temperature=0.0):
    """
    Generate a concise answer without any special tags.
    Uses greedy decoding (temperature=0) for deterministic output.
    """
    system_prompt = "You are a helpful medical assistant. Answer the user's question concisely and accurately."
    prompt = f"{system_prompt}\nUser: {question}\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False if temperature == 0 else True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract text after "Assistant:"
    if "Assistant:" in full_output:
        assistant_part = full_output.split("Assistant:")[-1]
    else:
        assistant_part = full_output
    return assistant_part.strip()

# Test examples
test_questions = [
    "What are the early symptoms of a heart attack?",
    "How can I treat a minor burn at home?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "Is it safe to take ibuprofen during pregnancy?",
    "How often should adults get a flu shot?"
]

print("\n" + "="*80)
print("TESTING FINETUNED BIOMEDICAL CHATBOT (NO THINK TAGS)")
print("="*80)

for i, q in enumerate(test_questions, 1):
    print(f"\n[{i}] Question: {q}")
    answer = generate_answer(q, max_new_tokens=120, temperature=0.0)
    print(f"Answer: {answer}")
    print("-"*80)